In [40]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Part 3: Collaborative Filtering using K-Nearest Neighbors
**Collaborative Filtering** is based on idea that "similar users like similar items". This algorithm using **K-Nearest Neighbors** algorithm and can be approached in two ways:
* **User-User CF:** Find users with similar rating histories to predict preferences of a user
* **Item-Item CF:** Find items that are rated similarly by same groups of users to recommed items related to what a user has already liked

## Import libraries

In [60]:
import pandas as pd 
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse 
from sklearn.metrics import mean_squared_error
import joblib

## 1.Import files

### Train-Test data (Utility Matrix)

In [42]:
ratings_train=pd.read_csv('../data/processed/ratings_b_train.csv',header=0).values
ratings_test=pd.read_csv('../data/processed/ratings_b_test.csv',header=0).values

In [43]:
ratings_train

array([[        0,         0,         5, 874965758],
       [        0,         1,         3, 876893171],
       [        0,         2,         4, 878542960],
       ...,
       [      942,      1187,         3, 888640250],
       [      942,      1227,         3, 888640275],
       [      942,      1329,         3, 888692465]], shape=(90570, 4))

In [44]:
n_users=len(np.unique(ratings_train[:,0]))
print(f'Number of users: {n_users}')

Number of users: 943


### Item-profiles data

In [45]:
items_profile=np.load('../data/processed/items_profile.npy')
items_profile

array([[0, 0, 1, ..., 0, 0, 0],
       [1, 1, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(1682, 18))

In [46]:
n_items=items_profile.shape[0]
print(f'Numbers of movies: {n_items}')

Numbers of movies: 1682


## 2.Build Collaborative Filtering Recommend System

### 2.1.Similarity functions
To find "neighbors", we use **Cosine Similarity**. Unlike Euclide distance, it measures **angle** between two vectors:
$$similarity(\mathbf{u_1},\mathbf{u_2})=\frac{\mathbf{u_1} \cdot \mathbf{u_2}}{||\mathbf{u_1}|| \cdot ||\mathbf{u_2}||}$$

### 2.2.User-User Collaborative Filtering

#### 2.2.1.Normalize data
We subtract each cell of utility matrix (not sparse) by averange of values by columns and replace missing value by 0. The reasons why we have to have the step because:
* Subtracting the averange of each columns result in both negative and positive values within each column. The positive values represent that a user likes a movie, nagative values represent that a user do not like a movie. Zero values represent that  an undetermined state of whether the user likes the item.
* An advantage of storing utility matrix under a sparse matrix that the dimensions of utility matrix are very large because of the huge number of users and items. Observed that the number of known data is very small compared to the size of utility matrix, so that is better to store the data under sparse matrix, ie just save non-zero values and those positive.

#### 2.2.2.Rating prediction

Popular function to predict rating of user $u$ for movie $i$:
$$\hat{y}_{i,u}=\frac{\sum_{u_j \in \mathcal{N}(u,i)} \bar{y}_{i,u_j} \text{sim}(u,u_j)}{\sum_{u_j \in \mathcal{N}(u,i)} |\text{sim}(u,u_j)|}$$
**Where**
* $\mathcal{N}(u,i)$: The set of k users in neighborhood (ie have highest **similarity**) of users $u$ who have rated movie $i$
* $\text{sim}(u,u_j)$: The **similarity score** between user $u$ and $u_j$ 
* $\bar{y}_{i,u_j}$: The rating of neighbor $u_j$ for movie $i$ (already mean-centered)

To translate normalized predictions back into a human-readable format, we must perform Denormalization. This is achieved by adding the user-specific mean ratings back to each corresponding column of the predicted matrix $\hat{Y}$.

### 2.3.Item-Item Collaborative Filtering 

#### 2.3.1.Analysis User-User Collaborative Filtering

* In fact, the number of users is very large compared to the number of items, so that the size of user-user similary matrix is absolutely big. This reason makes storing user-user matrix in some situations is very challenging.
* The utility matrix is usually sparse. Because the number of users is very large compared to the number of items, many columns of utility matrix are very sparse (ie just a few of elements is non-zero), the reason can be that the users are lazy to rate movies. So that as soon as u user change ratings or add new ratings, the mean of ratings and the normalized rating vector change significant, result in the computing Similarity matrix-actually waste much memory and time, the similarity matrix also recompute.

#### 2.3.2.Analysis Item-Item Collaborative Filtering

Conversely, calculating similarity between items and recommending those similar to a user's favorites offers the following key benefits:
* Efficiency in Storage and Computation: Since the number of items is much smaller than the number of users, the Similarity Matrix in this case is significantly smaller. This makes it far more efficient to store and speeds up subsequent computational steps.
* Model Stability: While the number of known entries in the Utility Matrix remains the same, there are fewer rows (items) than columns (users). Consequently, on average, each row contains more known ratings than each column. This is intuitive since a single item is often rated by many users. As a result, the mean value of each row is less sensitive to a few new ratings, meaning the Similarity Matrix stays stable and requires less frequent updates.

#### 2.3.3. Computing

Item-Item CF can be received from User-User CF by transposing Utility Matrix, treating items as users. After computing the result, we transpose again to receive the result

### 2.3.Define class for system

In [47]:
class NBCF:
    def __init__(self,n_users,n_items,k,dist_func=cosine_similarity,uuCF=1):
        self.uuCF=uuCF
        self.dist_func=dist_func
        self.Ybar=None
        self.k=k
        self.n_users=n_users if uuCF else n_items
        self.n_items=n_items if uuCF else n_users
    def normalize(self,data):
        self.Y_data=data if self.uuCF else data[:,[1,0,2]]
        self.Ybar=self.Y_data.copy()
        users=self.Y_data[:,0]
        self.mu=np.zeros((self.n_users,))
        for i in range(self.n_users):
            ids=np.where(users==i)[0].astype(int)
            ratings=self.Y_data[ids,2]
            m=np.mean(ratings)
            if np.isnan(m):
                m=0
            self.mu[i]=m
            self.Ybar[ids,2]=ratings-m
        self.Ybar=sparse.coo_matrix((self.Ybar[:,2],(self.Ybar[:,1],self.Ybar[:,0])),
                                    (self.n_items,self.n_users))
        self.Ybar=self.Ybar.tocsr()
        
    def similarity(self):
        self.S=self.dist_func(self.Ybar.T,self.Ybar.T)
    
    def fit(self,data):
        self.normalize(data)
        self.similarity()
    
    def _predict(self,u,i,normalized=1):
        ids=np.where(self.Y_data[:,1]==i)[0].astype(np.int32)
        users_rated_i=(self.Y_data[ids,0]).astype(np.int32)
        sim=self.S[u,users_rated_i]
        a=np.argsort(sim)[-self.k:]
        nearest=sim[a]
        r=self.Ybar[i,users_rated_i[a]]
        if normalized:
            return (r*nearest)[0]/(np.abs(nearest).sum()+1e-9)
        return (r*nearest)[0]/(np.abs(nearest).sum()+1e-9)+self.mu[u]
    
    def predict(self,u,i,normalized=1):
        if self.uuCF: return self._predict(u,i,normalized)
        return self._predict(i,u,normalized)
    
    def predict_test(self,ratings_test):
        n_tests=ratings_test.shape[0]
        prediction=np.zeros((n_tests,))
        for n in range(n_tests):
            prediction[n]=self.predict(ratings_test[n,0],ratings_test[n,1],normalized=0)
        return prediction
    
    def RMSE(self,data,predict):
        return np.sqrt(mean_squared_error(data[:,2],predict))
    
    def recommend_for_user(self,u,top=10):
        if self.uuCF:
            ids=np.where(self.Y_data[:,0]==u)[0].astype(np.int32)
            items_rated_by_u=self.Y_data[ids,1].tolist()
            n=self.n_items
        else:
            ids=np.where(self.Y_data[:,1]==u)[0].astype(np.int32)
            items_rated_by_u=self.Y_data[ids,0].tolist()
            n=self.n_users
        a=np.zeros((n,))
        a[items_rated_by_u]=-1e9
        for i in range(n):
            if i not in items_rated_by_u:
                a[i]=self.predict(u,i)
        recommended_items=np.argsort(a)[-top:][::-1]
        return recommended_items
    
    def recommend_simiar_items(self,i,top=10):
        if self.uuCF == 1:
            raise ValueError("Method only available for Item-Item CF (uuCF=0).")
        sim_scores=self.S[i]
        similar_items=np.argsort(sim_scores)[-(top+1):-1][::-1]
        return similar_items
            

## 3.Run and test model

### 3.1.User-User Model

In [48]:
uu_model=NBCF(n_users,n_items,k=30,uuCF=1)
uu_model.fit(ratings_train)

**Predict test**

In [49]:
prediction=uu_model.predict_test(ratings_test)
prediction

array([3.39312363, 3.76192262, 4.34459136, ..., 3.04243369, 3.247699  ,
       3.3734243 ], shape=(9430,))

**Evaluating model**

In [50]:
rmse=uu_model.RMSE(ratings_test,prediction)
print(f'RMSE: {rmse}')

RMSE: 0.9951644424076038


**Recommend**

In [51]:
movie_mapping=pd.read_csv('../data/processed/movie_mapping.csv',header=0)
movie_mapping

,movie id,movie title
0,0,Toy Story (1995)
1,1,GoldenEye (1995)
2,2,Four Rooms (1995)
3,3,Get Shorty (1995)
4,4,Copycat (1995)
...,...,...
1677,1677,Mat' i syn (1997)
1678,1678,B. Monkey (1998)
1679,1679,Sliding Doors (1998)
1680,1680,You So Crazy (1994)


In [53]:
recommend_movies_indices=uu_model.recommend_for_user(0) ## user 1
filter_movies=movie_mapping.loc[recommend_movies_indices,'movie title']
for index,title in filter_movies.items():
    print(f'Movie ID: {index+1}, Movie Title: {title}')

Movie ID: 1472, Movie Title: Visitors, The (Visiteurs, Les) (1993)
Movie ID: 1368, Movie Title: Mina Tannenbaum (1994)
Movie ID: 1643, Movie Title: Angel Baby (1995)
Movie ID: 1467, Movie Title: Saint of Fort Washington, The (1993)
Movie ID: 1500, Movie Title: Santa with Muscles (1996)
Movie ID: 814, Movie Title: Great Day in Harlem, A (1994)
Movie ID: 1642, Movie Title: Some Mother's Son (1996)
Movie ID: 1651, Movie Title: Spanish Prisoner, The (1997)
Movie ID: 1636, Movie Title: Brothers in Trouble (1995)
Movie ID: 1650, Movie Title: Butcher Boy, The (1998)


### 3.2.Item-Item Model

In [54]:
ii_model=NBCF(n_users,n_items,k=30,uuCF=0)
ii_model.fit(ratings_train)

E:\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
E:\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


**Predict test**

In [55]:
prediction_ii=ii_model.predict_test(ratings_test)
prediction_ii

array([3.18767553, 3.79016328, 4.44418971, ..., 2.99248979, 3.11318271,
       3.19429963], shape=(9430,))

**Evaluating model** 

In [56]:
rmse_ii=ii_model.RMSE(ratings_test,prediction_ii)
print(f'RMSE: {rmse_ii}')

RMSE: 0.9867689046410655


**Recommend movies for a user**

In [57]:
recommend_movies_indices=ii_model.recommend_for_user(0)
filter_movies=movie_mapping.loc[recommend_movies_indices,'movie title']
for index,title in filter_movies.items():
    print(f'Movie ID: {index+1}, Movie Title: {title}')

Movie ID: 870, Movie Title: Touch (1997)
Movie ID: 1609, Movie Title: B*A*P*S (1997)
Movie ID: 1600, Movie Title: Guantanamera (1994)
Movie ID: 1545, Movie Title: Frankie Starlight (1995)
Movie ID: 1527, Movie Title: Senseless (1998)
Movie ID: 1272, Movie Title: Talking About Sex (1994)
Movie ID: 1264, Movie Title: Nothing to Lose (1994)
Movie ID: 1085, Movie Title: Carried Away (1996)
Movie ID: 919, Movie Title: City of Lost Children, The (1995)
Movie ID: 1252, Movie Title: Contempt (Mépris, Le) (1963)


**Recommed similar movies based on ratings**

In [58]:
recommend_movies_indices=ii_model.recommend_simiar_items(0) ## movie 0
filter_movies=movie_mapping.loc[recommend_movies_indices,'movie title']
for index,title in filter_movies.items():
    print(f'Movie ID: {index+1}, Movie Title: {title}')

Movie ID: 95, Movie Title: Aladdin (1992)
Movie ID: 71, Movie Title: Lion King, The (1994)
Movie ID: 28, Movie Title: Apollo 13 (1995)
Movie ID: 588, Movie Title: Beauty and the Beast (1991)
Movie ID: 181, Movie Title: Return of the Jedi (1983)
Movie ID: 926, Movie Title: Down Periscope (1996)
Movie ID: 235, Movie Title: Mars Attacks! (1996)
Movie ID: 82, Movie Title: Jurassic Park (1993)
Movie ID: 204, Movie Title: Back to the Future (1985)
Movie ID: 96, Movie Title: Terminator 2: Judgment Day (1991)


## Save model

In [61]:
joblib.dump(ii_model, '../models/cf_model.pkl')

['../models/cf_model.pkl']